In [13]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model,after_model
from rich import print as rprint
from langgraph.runtime import Runtime
from langchain.messages import  RemoveMessage
from langgraph.graph.message import  REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


model = ChatOpenAI(
    model="MiniMax-M3",
    api_key='sk-cp-GsTRU_GlFRX_WPY8cf8IqBTDtrVJP4zl4EWLZ8D7627lrDtV-MbWYnWWK7rmvT7ZRluuMtUL7VP2sCD2uq6SzkpIlY0EAJtCIHW2uSGAT4QKYxX9mJQmK8I',
    base_url="https://api.minimaxi.com/v1",
)

# 策略---消息裁剪：在模型调用钱裁剪消息列表，控制模型可以看到的上下文范围
@before_model
def trim_messages(state:AgentState, runtime:Runtime):
    messages = state["messages"]
    if len(messages) <= 3:
        return None
    first_msg = messages[0]
    # 如果有偶数条消息则取最近3条，如果有奇数条消息则取最近4条
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages
    return {"messages": [
        RemoveMessage(id=REMOVE_ALL_MESSAGES),
        *new_messages
    ]}
    
# 策略---消息删除：强调模型调用完成后将消息从消息列表删除
@after_model
def delete_old_messages(state:AgentState, runtime:Runtime):
    messages = state["messages"]
    if len(messages) > 5:
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None




In [11]:
agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)

config = {
    "configurable":{
        "thread_id": "thread_1"
    }
}

agent.invoke({"messages":[HumanMessage(content="你好，我是老王")]}, config)
agent.invoke({"messages":[HumanMessage(content="从现在起，你叫小王")]}, config)
agent.invoke({"messages":[HumanMessage(content="今天天气不错")]}, config)
agent.invoke({"messages":[HumanMessage(content="今天的生意也不错")]}, config)
final_re = agent.invoke({"messages":[HumanMessage(content="告诉我，你是谁，我是谁")]}, config)

for msg in final_re["messages"]:
    msg.pretty_print()


================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

<think>The user is making a casual conversational remark about the weather being nice today. I should respond naturally and friendly, staying in character as 小王.</think>

是啊，老王！好天气总是让人心情舒畅 ☀️

不知道你今天有什么计划？出去散散步、晒晒太阳都是不错的选择。要是无聊的话，小王也可以陪你聊聊天、解答问题，或者帮你规划一下今天的安排，怎么样？
================================ Human Message =================================

今天的生意也不错
================================== Ai Message ==================================

<think>The user is talking about their business being good today. I should respond in a friendly, congratulatory way, expressing happiness for them. Keep it natural and casual.</think>

那真是双喜临门呀，老王！🎉 好天气加上好生意，今天简直完美~

不知道老王是做什么生意的呀？小王虽然帮不上什么实际的忙，但如果你有什么需要整理思路、记录点子、或者起草文件之类的，我可以随时效劳！
================================ Human Message =================================

告诉我，你是谁，我是谁
============

In [18]:
agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config = {
    "configurable":{
        "thread_id": "thread_2"
    }
}

agent.invoke({"messages":[HumanMessage(content="你好，我是老王")]}, config)
agent.invoke({"messages":[HumanMessage(content="从现在起，你叫小王")]}, config)
agent.invoke({"messages":[HumanMessage(content="今天天气不错")]}, config)
agent.invoke({"messages":[HumanMessage(content="今天的生意也不错")]}, config)
final_re = agent.invoke({"messages":[HumanMessage(content="告诉我，你是谁，我是谁")]}, config)

for msg in final_re["messages"]:
    msg.pretty_print()


================================== Ai Message ==================================

<think>用户说"今天天气不错"，这是一句闲聊。我作为"小王"应该自然地回应，保持友好轻松的聊天氛围。不过需要注意，我没有实时天气数据，所以我不能假装知道用户所在地的天气情况。我可以表达认同这个说法，但不要编造具体的天气信息。</think>

是啊，天气好心情也会跟着好起来！☀️

不知道老王是出门踏青了，还是在家里享受悠闲的一天呢？小王虽然看不到外面的天气，但能感受到你这份好心情！😊
================================ Human Message =================================

今天的生意也不错
================================== Ai Message ==================================
Name: MiniMax AI

<think>
用户说"今天的生意也不错"，这是在分享他的生意状况。作为小王，我应该以友好、关心的方式回应，祝福他生意兴隆。可以表现出为他高兴的样子，同时也可以好奇地问一下他的生意是做什么的。
</think>
哇，那真是双喜临门啊！天气好，生意也好，老王今天肯定顺风顺水！💰✨

不知道老王做的是哪方面的生意呀？做买卖最讲究的就是天时地利人和，今天看来全都占了！恭喜恭喜！🎉
================================ Human Message =================================

告诉我，你是谁，我是谁
================================== Ai Message ==================================
Name: MiniMax AI

<think>
用户问我两个问题：
1. 我是谁
2. 用户是谁

根据之前的对话，我（小王）已经知道用户是"老王"。用户要求我把自己的名字改成"小王"。

所以我的回答应该是：
- 我是小王
- 用户是老王

简洁明了地回答这两个问题。
</think>
我是**小王**，你的小帮手呀！😊

